**© Copyright AIDENTIFY. All rights reserved.**

# Part 1 | Session 7: microGPT 실습 (Karpathy 2026, 의존성 없는 243줄 GPT)

Andrej Karpathy가 2026년 2월에 공개한 **microGPT** ([gist.github.com/karpathy/8627fe...](https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95))는 GPT의 학습/추론 알고리즘 전체를 **순수 Python 243줄, 외부 의존성 0**으로 구현한 교육용 아트 프로젝트입니다.

PyTorch / NumPy / TensorFlow 없이, `os`·`math`·`random`만으로:
- 자체 **autograd 엔진** (`Value` 클래스, ~30줄)
- GPT-2 스타일 **Transformer 아키텍처** (RMSNorm, multi-head attention, MLP, residual)
- **Adam 옵티마이저** (1차/2차 모멘트, bias correction, lr decay)
- **학습 루프** + **autoregressive 생성**

을 모두 한 파일에 담았습니다. Karpathy 본인이 "This file is the complete algorithm. Everything else is just efficiency." 라고 말한 그대로입니다.

### 학습 목표
- GPT의 **알고리즘적 본질**을 파이썬 한 파일로 끝까지 따라가본다 (블랙박스 ZERO).
- 자체 autograd로 **chain rule이 어떻게 그래프를 거꾸로 흐르는지** 직접 확인한다.
- Multi-head attention, residual, RMSNorm 등 핵심 블록을 **PyTorch 추상화 없이** 구현한다.
- Adam의 1차/2차 모멘트 업데이트 식을 **코드 한 줄 한 줄**로 본다.
- 본 실습에서는 **한국 이름 생성** 태스크로 한글에서도 동일 알고리즘이 작동함을 시연한다.

### 참고
- 원본 Gist: https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95
- 미러 페이지: https://karpathy.ai/microgpt.html
- 블로그 해설: http://karpathy.github.io/2026/02/12/microgpt/

---
## 점진적 복잡도: train0 → train5

Karpathy의 블로그는 microGPT를 **6단계의 점진적 복잡도**로 설명합니다. 본 노트북은 최종 형태(train5 = microgpt)를 한 번에 다루지만, 각 섹션이 어느 단계에 해당하는지 표시해두었습니다.

| 단계 | 추가되는 개념 | 본 노트북 섹션 |
|------|---------------|----------------|
| `train0.py` | Bigram count table (NN 없음) | (개념만 소개) |
| `train1.py` | MLP + 수동 gradient + SGD | (개념만 소개) |
| `train2.py` | 자체 autograd (`Value`) 도입 | §3 Autograd |
| `train3.py` | Positional embedding + single-head attention + RMSNorm + residual | §4–6 모델 |
| `train4.py` | Multi-head + 여러 layer (= 전체 GPT) | §4–6 모델 |
| `train5.py` | Adam optimizer (= 완성판 microgpt) | §7 학습 |

각 단계가 **무엇을 더해야 다음 단계로 갈 수 있는지** 의식하면서 읽으면, 현대 LLM의 모든 구성요소가 왜 필요한지가 자연스럽게 보입니다.

---
## 1️⃣ 셋업 (의존성 없음)

`pip install` 단계가 없습니다. Python 표준 라이브러리만 씁니다.
- `os` — 파일 시스템
- `math` — `log`, `exp`
- `random` — seed, sampling, gaussian init

In [ ]:
import os
import math
import random

random.seed(42)  # Let there be order among chaos
print('표준 라이브러리만 import 완료. PyTorch / NumPy 없음.')

---
## 2️⃣ 데이터셋 & 토크나이저: **한국 이름 + 자모 분해**

GPT는 **next-token prediction (causal LM)** 모델입니다. 이번 실습에서는 이 능력을 한글로 시연합니다.

### 왜 자모(jamo) 분해인가?

한글 음절은 약 **11,172개** 입니다 (가, 각, 간, 갇, 갈, ...). 음절을 그대로 토큰으로 쓰면 vocab이 폭발 → 초경량 GPT가 학습 불가능.

**해결**: 한글 음절은 항상 **초성(19) + 중성(21) + 종성(27 + 없음)** 으로 분해됩니다.

```
민준  →  ㅁ ㅣ ㄴ   ㅈ ㅜ ㄴ        (6개 자모)
서연  →  ㅅ ㅓ   ㅇ ㅕ ㄴ           (5개 자모)
```

이 트릭으로 **vocab ≈ 25** 으로 줄어들어 microGPT 스케일에 딱 맞습니다.

### 학습 태스크: **이름 부분만** 생성

**중요**: 한국 성씨는 약 300개로 정해진 집합 (김, 이, 박, 최, ...) 이므로 **생성 대상이 아닙니다**. 새 성씨를 만드는 건 의미 없음.

따라서:
- **학습**: GPT가 ~80개 흔한 한국 두 글자 **이름** (민준, 서연, 지호, 하은, ...) 의 자모 패턴만 학습
- **추론**: 
  1. **성씨**는 실제 한국 성씨 목록 (`SURNAMES`) 에서 `random.choice()` 로 추첨
  2. **이름**은 GPT가 자모 단위로 생성 → 한글로 합성
  3. 둘을 합쳐서 풀네임 (`성씨 + 생성된 이름`) 출력

### 모델이 학습하는 것 (이름에 한정)

1. **한글 음절 구조**: 자음(초성) → 모음(중성) → (자음 종성) 순서
2. **이름 첫 음절 분포**: 민, 서, 지, 하, 예, 윤 등 흔한 시작 자모
3. **이름 끝 자모**: -준, -은, -아, -희, -원 같은 흔한 이름 종성
4. **음운 조화**: 양성 모음끼리, 음성 모음끼리 어울리는 경향

### 기대 결과

성씨 영역의 무작위성은 사라지고 (실제 성씨 풀에서만 추첨), 이름 영역에서 **학습 데이터에 없던 새 이름** 이 생성됩니다. 예: 김지윤(학습 데이터), 박서연(학습 데이터) → 박**서린**(새 조합), 정**유원**(새 조합).

In [ ]:
# --- 한글 자모 분해 / 합성 유틸 (외부 라이브러리 없음) ---
CHO_LIST  = 'ㄱㄲㄴㄷㄸㄹㅁㅂㅃㅅㅆㅇㅈㅉㅊㅋㅌㅍㅎ'                              # 19개
JUNG_LIST = 'ㅏㅐㅑㅒㅓㅔㅕㅖㅗㅘㅙㅚㅛㅜㅝㅞㅟㅠㅡㅢㅣ'                          # 21개
JONG_LIST = ' ㄱㄲㄳㄴㄵㄶㄷㄹㄺㄻㄼㄽㄾㄿㅀㅁㅂㅄㅅㅆㅇㅈㅊㅋㅌㅍㅎ'              # 28개 (idx 0 = 종성 없음)

def decompose(text):
    """한글 문자열 → 자모 리스트.  '김민준' -> ['ㄱ','ㅣ','ㅁ','ㅁ','ㅣ','ㄴ','ㅈ','ㅜ','ㄴ']"""
    out = []
    for ch in text:
        code = ord(ch) - 0xAC00
        if 0 <= code < 11172:                          # 한글 음절 영역
            cho  = code // 588
            jung = (code % 588) // 28
            jong = code % 28
            out.append(CHO_LIST[cho])
            out.append(JUNG_LIST[jung])
            if jong > 0:
                out.append(JONG_LIST[jong])
        else:
            out.append(ch)
    return out

def compose(jamo_list):
    """자모 리스트 → 한글 문자열.  ['ㄱ','ㅣ','ㅁ','ㅁ','ㅣ','ㄴ','ㅈ','ㅜ','ㄴ'] -> '김민준'.

    규칙: 초성+중성+(다음이 종성 후보이고 그 다음에 모음이 안 오면 종성으로 흡수).
    잘못된 자모 시퀀스는 그대로 출력 (예: 모음만 단독).
    """
    out = []
    i, n = 0, len(jamo_list)
    while i < n:
        if i+1 < n and jamo_list[i] in CHO_LIST and jamo_list[i+1] in JUNG_LIST:
            cho_idx  = CHO_LIST.index(jamo_list[i])
            jung_idx = JUNG_LIST.index(jamo_list[i+1])
            jong_idx = 0
            consume  = 2
            # 다음 자모가 종성 후보(자음)이고, 그 뒤가 비었거나 또 자음이면 → 현재 음절의 종성
            if i+2 < n and jamo_list[i+2] in JONG_LIST[1:]:
                if i+3 >= n or jamo_list[i+3] not in JUNG_LIST:
                    jong_idx = JONG_LIST.index(jamo_list[i+2])
                    consume  = 3
            code = 0xAC00 + cho_idx * 588 + jung_idx * 28 + jong_idx
            out.append(chr(code))
            i += consume
        else:
            out.append(jamo_list[i])
            i += 1
    return ''.join(out)

# 검증
assert decompose('김민준') == ['ㄱ','ㅣ','ㅁ','ㅁ','ㅣ','ㄴ','ㅈ','ㅜ','ㄴ']
assert compose(['ㄱ','ㅣ','ㅁ','ㅁ','ㅣ','ㄴ','ㅈ','ㅜ','ㄴ']) == '김민준'
assert compose(decompose('한국어 자연스럽다')) == '한국어 자연스럽다'
print('자모 분해/합성 유틸 정의 완료. (외부 라이브러리 없음)')
print(f"예시: '김민준' → {decompose('김민준')}")
print(f"예시: 다시 합성 → '{compose(decompose('김민준'))}'")

In [ ]:
# --- 한국 성씨 목록 (통계청 2015 인구조사 기준, 한글 중복 제거) ---
# 생성하지 않음. 추론 시 random.choice() 로 추첨용.
SURNAMES = [
    # 인구 100만+ 상위 9성
    '김','이','박','최','정','강','조','윤','장',
    # 인구 10만+
    '임','한','오','서','신','권','황','안','송','류',
    '전','홍','고','문','양','손','배','백','허','유',
    '남','심','노','하','곽','성','차','주','우','구',
    # 인구 1만+
    '민','나','진','지','엄','채','원','천','방','공',
    '현','함','변','염','여','추','도','소','석','선',
    '설','마','길','연','위','표','명','기','반','왕',
    '금','옥','육','인','맹','제','모','탁','국','어',
    # 인구 1천+
    '경','가','빈','사','상','단','두','견','시','종',
    '음','봉','매','평','풍','점','좌','묘','형','운',
    '학','곡','청','환','등','부','즙','개','비','옹',
    '간','동','교','복','용','승','호','삼','순','피',
    # 두 글자 복성 (compound surnames)
    '남궁','황보','제갈','사공','서문','선우','독고','동방',
]

# --- 학습 데이터: 흔한 한국 두 글자 '이름' (성씨 제외) ---
GIVEN_NAMES = [
    # 흔한 남자 이름
    '민준','서준','도윤','예준','시우','주원','하준','지호','준우','건우',
    '현우','우진','선우','연우','정우','승우','시윤','유준','은우','지환',
    '지원','지훈','동현','민재','현준','도현','시현','윤호','진우','태윤',
    '재준','이준','민호','수호','석현','수현','지석','동민','준영','상민',
    # 흔한 여자 이름
    '서연','서윤','지우','서현','민서','하은','하윤','윤서','지유','지민',
    '채원','수아','다은','예은','지아','소율','예린','시은','유나','예원',
    '채은','시아','가은','수빈','유진','윤아','다인','채윤','나윤','다현',
    '은서','지윤','민지','지수','다영','수정','지혜','민영','수연','은지',
]

# 학습 데이터는 '이름'만 (성씨는 학습 X, inference 시점에 추첨)
training_names = list(GIVEN_NAMES)
random.shuffle(training_names)

# 자모 분해 → docs (list of list[str])
docs = [decompose(g) for g in training_names]

# 중복 확인 (한글 기준)
assert len(SURNAMES) == len(set(SURNAMES)), '성씨 목록에 중복이 있습니다'
n_single = sum(1 for s in SURNAMES if len(s) == 1)
n_double = sum(1 for s in SURNAMES if len(s) == 2)

print(f'한국 성씨 목록 (생성 X)   : 총 {len(SURNAMES)}개')
print(f'  - 한 글자 성씨           : {n_single}개')
print(f'  - 두 글자 복성           : {n_double}개')
print(f'  예시 (상위 10)           : {SURNAMES[:10]}')
print(f'  예시 (희성)              : {SURNAMES[80:90]}')
print(f'  복성                     : {[s for s in SURNAMES if len(s) == 2]}')
print(f'\n학습 데이터 (이름) 수    : {len(training_names)}')
print(f'\n학습 이름 예시 (분해 결과):')
for name, doc in list(zip(training_names, docs))[:5]:
    print(f"  {name} → {doc}")
print(f'\n자모 시퀀스 길이 분포: 최소 {min(len(d) for d in docs)}, 최대 {max(len(d) for d in docs)}, 평균 {sum(len(d) for d in docs)/len(docs):.1f}')
print(f'\n참고: 가능한 풀네임 후보 = {len(SURNAMES)} × {len(GIVEN_NAMES)} = {len(SURNAMES)*len(GIVEN_NAMES):,} 개')
print('       (생성 시 이름이 학습 데이터 외면 [NEW] 표시)')

In [ ]:
# --- 토크나이저: 데이터에 등장한 자모만 모아 vocab 구성 ---
uchars = sorted({jamo for doc in docs for jamo in doc})  # 데이터에 실제 등장하는 자모만
BOS = len(uchars)                # 마지막 인덱스를 BOS 토큰으로
vocab_size = len(uchars) + 1

print(f'unique 자모 ({len(uchars)}개): {uchars}')
print(f'BOS token id: {BOS}')
print(f'vocab size: {vocab_size}')

# 토큰화 데모
sample_name = training_names[0]
sample_doc  = docs[0]
encoded = [uchars.index(j) for j in sample_doc]
decoded = compose([uchars[i] for i in encoded])
print(f"\n예시 이름: '{sample_name}'")
print(f"  자모 분해: {sample_doc}")
print(f"  토큰 ID  : {encoded}")
print(f"  복원    : '{decoded}'")

---
## 3️⃣ Autograd: `Value` 클래스 (train2 단계)

PyTorch의 `Tensor` + `autograd` 를 30줄 파이썬으로 직접 만듭니다.

**핵심 아이디어** — 모든 연산이 새 `Value` 노드를 만들면서 다음 두 가지를 함께 기록한다:
1. `_children`: 이 노드가 어떤 입력 노드들로부터 만들어졌는지 (computation graph의 부모)
2. `_local_grads`: 출력에 대한 각 자식의 **local 미분값** (chain rule의 한 항)

그러면 `backward()` 는 **위상정렬된 그래프를 역순으로 순회**하면서 `child.grad += local_grad * v.grad` 한 줄로 chain rule을 자동 적용합니다.

In [ ]:
# Let there be Autograd to recursively apply the chain rule through a computation graph
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')  # Python optimization for memory usage

    def __init__(self, data, children=(), local_grads=()):
        self.data = data                # scalar value of this node calculated during forward pass
        self.grad = 0                   # derivative of the loss w.r.t. this node, calculated in backward pass
        self._children = children       # children of this node in the computation graph
        self._local_grads = local_grads # local derivative of this node w.r.t. its children

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, other): return Value(self.data**other, (self,), (other * self.data**(other-1),))
    def log(self):  return Value(math.log(self.data), (self,), (1/self.data,))
    def exp(self):  return Value(math.exp(self.data), (self,), (math.exp(self.data),))
    def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
    def __neg__(self):     return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other):  return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other):  return self * other**-1
    def __rtruediv__(self, other): return other * self**-1

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad

print('Value 클래스 정의 완료. 이제 모든 연산이 computation graph를 자동으로 쌓습니다.')

### Autograd 동작 확인

간단한 식 `f = (a + b) * c` 에서 손으로 계산한 미분값과 `backward()` 결과를 비교합니다.

- `df/da = c`,  `df/db = c`,  `df/dc = (a + b)`
- `a=2, b=3, c=4` 일 때:  `df/da = 4`, `df/db = 4`, `df/dc = 5`

In [ ]:
a = Value(2.0)
b = Value(3.0)
c = Value(4.0)
f = (a + b) * c
f.backward()

print(f'forward: f = (a + b) * c = ({a.data} + {b.data}) * {c.data} = {f.data}')
print(f'backward: df/da = {a.grad}  (expected 4)')
print(f'          df/db = {b.grad}  (expected 4)')
print(f'          df/dc = {c.grad}  (expected 5)')
assert (a.grad, b.grad, c.grad) == (4, 4, 5), 'autograd 검증 실패'

---
## 4️⃣ 모델 파라미터 초기화 (train3–4 단계)

GPT-2 구조의 미니 버전으로 다음 파라미터들을 만듭니다:
- `wte` (token embedding): `[vocab_size, n_embd]`
- `wpe` (position embedding): `[block_size, n_embd]`
- `lm_head` (출력 projection): `[vocab_size, n_embd]`
- 각 layer마다: `attn_wq, attn_wk, attn_wv, attn_wo, mlp_fc1, mlp_fc2`

**GPT-2와의 차이 (Karpathy 본인 주석):**
- LayerNorm → **RMSNorm** (더 간결)
- bias 없음
- GeLU → **ReLU** (autograd 구현이 더 간단)

In [ ]:
# Initialize the parameters, to store the knowledge of the model
n_layer = 1     # depth of the transformer neural network (number of layers)
n_embd = 16     # width of the network (embedding dimension)
block_size = 12 # max context length (한국 이름 자모 시퀀스: 4~9 + BOS 양쪽 → 최대 11)
n_head = 4      # number of attention heads
head_dim = n_embd // n_head  # derived dimension of each head

matrix = lambda nout, nin, std=0.08: [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]

state_dict = {
    'wte':     matrix(vocab_size, n_embd),
    'wpe':     matrix(block_size, n_embd),
    'lm_head': matrix(vocab_size, n_embd),
}
for i in range(n_layer):
    state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)

params = [p for mat in state_dict.values() for row in mat for p in row]
print(f'num params: {len(params)}')
print(f'(비교) GPT-2 small: ~124M, GPT-3: ~175B')

---
## 5️⃣ 모델 컴포넌트

Transformer를 구성하는 세 개의 작은 함수를 정의합니다.
- `linear(x, w)` — 행렬-벡터 곱 (bias 없음)
- `softmax(logits)` — numerical-stable softmax (max 빼기 트릭)
- `rmsnorm(x)` — Root Mean Square Layer Normalization

주의: 모든 연산이 `Value` 객체 위에서 일어나므로 **forward pass가 곧 computation graph 구축**입니다.

In [ ]:
def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

def softmax(logits):
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]

def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]

print('linear, softmax, rmsnorm 정의 완료.')

---
## 6️⃣ GPT forward pass (전체 아키텍처)

한 번의 호출로 **하나의 토큰**(현재 위치)에 대해 다음 토큰의 logits을 반환합니다.

**왜 한 토큰씩?** KV cache (`keys`, `values`)에 과거 토큰의 key/value를 누적하면서, **autoregressive하게** 한 토큰씩 진행합니다. 학습/추론 모두 동일한 함수를 사용합니다.

**구조 (각 layer):**
```
x → RMSNorm → Q,K,V → multi-head attention → linear → +residual
  → RMSNorm → fc1 → ReLU → fc2 → +residual
```

In [ ]:
def gpt(token_id, pos_id, keys, values):
    tok_emb = state_dict['wte'][token_id]                     # token embedding
    pos_emb = state_dict['wpe'][pos_id]                       # position embedding
    x = [t + p for t, p in zip(tok_emb, pos_emb)]             # joint token + position embedding
    x = rmsnorm(x)  # not redundant: needed for backward via the residual connection

    for li in range(n_layer):
        # 1) Multi-head Attention block
        x_residual = x
        x = rmsnorm(x)
        q = linear(x, state_dict[f'layer{li}.attn_wq'])
        k = linear(x, state_dict[f'layer{li}.attn_wk'])
        v = linear(x, state_dict[f'layer{li}.attn_wv'])
        keys[li].append(k)     # KV cache: append current k
        values[li].append(v)   # KV cache: append current v
        x_attn = []
        for h in range(n_head):
            hs = h * head_dim
            q_h = q[hs:hs+head_dim]
            k_h = [ki[hs:hs+head_dim] for ki in keys[li]]
            v_h = [vi[hs:hs+head_dim] for vi in values[li]]
            attn_logits  = [sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5 for t in range(len(k_h))]
            attn_weights = softmax(attn_logits)
            head_out = [sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h))) for j in range(head_dim)]
            x_attn.extend(head_out)
        x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
        x = [a + b for a, b in zip(x, x_residual)]

        # 2) MLP block
        x_residual = x
        x = rmsnorm(x)
        x = linear(x, state_dict[f'layer{li}.mlp_fc1'])
        x = [xi.relu() for xi in x]
        x = linear(x, state_dict[f'layer{li}.mlp_fc2'])
        x = [a + b for a, b in zip(x, x_residual)]

    logits = linear(x, state_dict['lm_head'])
    return logits

print('gpt() forward 정의 완료.')

---
## 7️⃣ Adam 옵티마이저 + 학습 루프 (train5 단계)

**Adam 업데이트 식 (Karpathy 코드 그대로):**
```
m_t = β1·m_{t-1} + (1-β1)·g_t            # 1차 모멘트 (gradient 평균)
v_t = β2·v_{t-1} + (1-β2)·g_t²           # 2차 모멘트 (gradient² 평균)
m̂_t = m_t / (1 - β1^t)                   # bias correction
v̂_t = v_t / (1 - β2^t)
θ_t -= lr_t · m̂_t / (√v̂_t + ε)
```

⚠️ **실행 시간** — 순수 Python autograd 특성상 느립니다. 한국 이름 자모 시퀀스(평균 ~7 자모) 기준:
- `num_steps=300`: 약 2~4분 (학습 추세 확인 가능)
- `num_steps=800`: 약 6~10분 (이름같은 한글 출력 시작)
- `num_steps=1500`: 약 12~20분 (안정적인 결과)

처음에는 300으로 시작해서 §8에서 출력 자모가 한글로 잘 합성되는지 확인 후, 결과가 마음에 안 들면 늘리세요.

In [ ]:
# Let there be Adam, the blessed optimizer and its buffers
learning_rate, beta1, beta2, eps_adam = 0.01, 0.85, 0.99, 1e-8
m = [0.0] * len(params)  # first moment buffer
v = [0.0] * len(params)  # second moment buffer

num_steps = 300  # 데모용. 결과가 약하면 800 또는 1500으로 늘리세요.
loss_history = []

for step in range(num_steps):
    # Take single document (= one Korean name, already a list of jamo), surround with BOS on both sides
    doc = docs[step % len(docs)]
    tokens = [BOS] + [uchars.index(j) for j in doc] + [BOS]
    n = min(block_size, len(tokens) - 1)

    # Forward the token sequence through the model, building up the computation graph all the way to the loss
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    losses = []
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits)
        loss_t = -probs[target_id].log()
        losses.append(loss_t)
    loss = (1 / n) * sum(losses)  # final average loss over the document sequence

    # Backward the loss, calculating the gradients with respect to all model parameters
    loss.backward()

    # Adam optimizer update
    lr_t = learning_rate * (1 - step / num_steps)  # linear learning rate decay
    for i, p in enumerate(params):
        m[i] = beta1 * m[i] + (1 - beta1) * p.grad
        v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
        m_hat = m[i] / (1 - beta1 ** (step + 1))
        v_hat = v[i] / (1 - beta2 ** (step + 1))
        p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
        p.grad = 0

    loss_history.append(loss.data)
    if (step + 1) % 20 == 0 or step == 0:
        print(f'step {step+1:4d} / {num_steps:4d} | loss {loss.data:.4f}')

print('\n학습 완료!')

In [ ]:
# 학습 곡선
# batch size = 1 (문서 하나씩) 이라 step별 loss는 매우 들쭉날쭉합니다.
# 진짜 추세를 보려면 raw 위에 이동평균(rolling mean)을 함께 그려야 합니다.
import matplotlib.pyplot as plt

def rolling_mean(xs, window):
    out = []
    for i in range(len(xs)):
        lo = max(0, i - window + 1)
        out.append(sum(xs[lo:i+1]) / (i - lo + 1))
    return out

window = max(5, len(loss_history) // 20)
ma = rolling_mean(loss_history, window)

# 구간별 평균(앞 20% / 뒤 20%) 으로 학습 효과를 수치로 비교
n = len(loss_history)
front = loss_history[: n // 5]
back  = loss_history[-n // 5 :]
front_avg = sum(front) / len(front)
back_avg  = sum(back)  / len(back)
baseline  = math.log(vocab_size)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(loss_history, alpha=0.25, color='steelblue', label='raw loss (batch=1, noisy)')
ax.plot(ma, color='crimson', linewidth=2.0, label=f'rolling mean (window={window})')
ax.axhline(y=baseline, color='gray', linestyle='--', alpha=0.6,
           label=f'ln(vocab_size) = {baseline:.2f} (uniform baseline)')
ax.axhline(y=front_avg, color='orange', linestyle=':', alpha=0.7,
           label=f'first 20% avg = {front_avg:.3f}')
ax.axhline(y=back_avg,  color='green',  linestyle=':', alpha=0.7,
           label=f'last 20% avg  = {back_avg:.3f}')
ax.set_xlabel('Step')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('microGPT Training Curve (raw + rolling mean)')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

drop = front_avg - back_avg
ppl_front, ppl_back = math.exp(front_avg), math.exp(back_avg)
print(f'앞 20% 평균 loss: {front_avg:.3f}  (perplexity {ppl_front:.1f})')
print(f'뒤 20% 평균 loss: {back_avg:.3f}  (perplexity {ppl_back:.1f})')
print(f'감소량        : {drop:+.3f} nats   → 다음 토큰 후보가 {ppl_front:.0f}개에서 {ppl_back:.0f}개로 좁혀짐')
print()
print('해석:')
print('- batch size = 1 이라 step별 loss는 어떤 문서를 뽑았느냐에 따라 들쭉날쭉합니다.')
print('- 빨간 rolling mean이 우하향이면 학습 진행 중. uniform baseline에 가까우면 학습 안 됨.')

---
## 8️⃣ 추론: 한국 이름 생성 (자모 → 한글 합성)

학습된 모델로 **BOS 토큰부터 시작해 한 자모씩** 샘플링한 다음, 자모 시퀀스를 한글로 합성합니다.
- BOS 토큰이 다시 나오면 종료 (= 이름 완성)
- `temperature` 가 작을수록 결정적 (자주 본 패턴)
- `temperature` 가 클수록 다양 (드문 패턴도 시도)

**판단 기준 (학습 정도)**:

| 출력 모습 | 학습 단계 |
|-----------|-----------|
| 자모가 무작위 (예: ㄱㅋㅋㄴㅁ) | 학습 안 됨 |
| 자모는 한글 음절로 합성되지만 어색 (예: 핅쥹) | 음절 구조만 학습 |
| **이름같은 한글** (예: 김지원, 박서연, 정하준) | **충분히 학습됨** |
| 학습 데이터에 없는 새 이름 (예: 정유린, 최하원) | 일반화까지 됨 |

In [ ]:
# Inference: 성씨는 실제 목록에서 추첨, 이름은 GPT가 자모 단위로 생성
temperature = 0.5  # 0.3~0.5 권장 (낮을수록 학습 패턴 충실)

train_given_set = set(training_names)   # 학습된 이름 집합

print(f'--- inference: 성씨 추첨 + 이름 생성 (temperature={temperature}, 30개 샘플) ---\n')
n_total = 0
n_valid = 0
n_novel = 0
for sample_idx in range(30):
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    token_id = BOS
    sample = []
    for pos_id in range(block_size):
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax([l / temperature for l in logits])
        token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
        if token_id == BOS:
            break
        sample.append(uchars[token_id])
    given   = compose(sample)
    surname = random.choice(SURNAMES)
    fullname = surname + given

    n_total += 1
    is_valid_hangul = bool(given) and all(0xAC00 <= ord(c) < 0xD7A4 for c in given)
    is_novel        = is_valid_hangul and (given not in train_given_set)
    if is_valid_hangul: n_valid += 1
    if is_novel       : n_novel += 1

    tag = '[NEW]' if is_novel else ('[OK ]' if is_valid_hangul else '[BAD]')
    print(f'  {sample_idx+1:2d}: {tag} {fullname:6s}    (성씨 "{surname}" 추첨 + 생성이름 "{given}")')

print(f'\n=== 학습 정도 측정 (이름 부분만 평가) ===')
print(f'전체 생성        : {n_total}')
print(f'유효 한글 음절   : {n_valid} / {n_total}  ({n_valid*100//n_total}%)  ← 음절 구조 학습')
print(f'학습데이터 외 이름: {n_novel} / {n_total}  ({n_novel*100//n_total}%)  ← 새 조합 생성 능력 ([NEW])')

### Temperature 비교

동일 모델로 temperature만 바꿔 생성 결과의 다양성/안정성을 비교합니다.

In [ ]:
for temp in [0.3, 0.5, 0.8, 1.2]:
    print(f'\n--- temperature = {temp} (성씨는 무작위 추첨) ---')
    for sample_idx in range(8):
        keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
        token_id = BOS
        sample = []
        for pos_id in range(block_size):
            logits = gpt(token_id, pos_id, keys, values)
            probs = softmax([l / temp for l in logits])
            token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
            if token_id == BOS:
                break
            sample.append(uchars[token_id])
        given   = compose(sample)
        surname = random.choice(SURNAMES)
        print(f'  {sample_idx+1}: {surname}{given}')

---
## 9️⃣ 같은 코드, 다른 데이터: 영화 제목 생성기

원본 microGPT의 핵심 디자인 — `docs = [line.strip() for line in open('input.txt')]` **한 줄만 바꾸면 도메인 전환이 끝난다.**

여기서는 위에서 만든 `Value` / `linear` / `softmax` / `rmsnorm` / `gpt(...)` / Adam 학습 루프를 **한 줄도 안 바꾸고**, 학습 데이터만 한국 영화 제목으로 갈아끼웁니다.

| 변수 | 이름 학습 시 | **영화 학습 시 (이번 섹션)** |
|------|--------------|------------------------------|
| `docs` | 한국 이름 자모 시퀀스 (~80개) | 영화 제목 자모 시퀀스 (~80편) |
| `uchars`, `BOS`, `vocab_size` | 자모 ~30개 | 자모 ~50개 |
| `block_size` | 12 | **24** (제목이 더 김) |
| `state_dict`, `params` | 위 vocab/block_size 기반 초기화 | 새 vocab/block_size로 **재초기화** |
| `Value`, `gpt()`, 학습/추론 코드 | (정의됨) | **그대로 재사용** |

> 핵심 메시지: **모델 코드는 도메인-독립적이다.** 데이터·vocab만 바꾸면 같은 알고리즘이 다른 도메인의 패턴을 학습한다 — 슬라이드(p.13) "LLM은 Scale의 게임" 메시지의 도메인 전환 미니 시연.

> 추론 시 4b의 "성씨 추첨" 같은 트릭은 없습니다. 처음(BOS)부터 끝(BOS)까지 모델이 직접 자모 시퀀스를 생성합니다.


In [ ]:
# 9-1. 학습 데이터: 한국 영화 제목 ~80편
MOVIE_TITLES = [
    # 2글자
    '곡성','괴물','마더','명량','암살','관상','사도','군도','타짜','광해',
    '하녀','명당','헌트','한산','독전','미옥','마녀','협녀','화이','바람',
    '이끼','황해','반도','인질','동주','박열','항거','럭키','박쥐',
    # 3-4글자
    '안시성','브로커','말모이','베를린','기생충','추격자','부산행','신세계',
    '도가니','변호인','아저씨','공조','강철비','극한직업','엑시트','도둑들',
    '왕의남자','무뢰한','늑대소년','써니','국제시장','베테랑','설국열차',
    '남한산성','신의한수','승리호','모가디슈','내부자들','검사외전',
    '청년경찰','비열한거리','범죄도시','택시운전사','검은사제들','봉오동전투',
    '신과함께','신라의달밤','건축학개론',
    # 5-6글자
    '헤어질결심','비상선언','미녀는괴로워','친절한금자씨','과속스캔들',
    '수상한그녀','더테러라이브','끝까지간다','복수는나의것','인천상륙작전',
    '범죄와의전쟁','낙원의밤','쓰리몬스터','관능의법칙','곤지암','7번방의선물',
]
MOVIE_TITLES = list(dict.fromkeys(MOVIE_TITLES))   # 중복 제거 (순서 보존)
random.shuffle(MOVIE_TITLES)

# 변수명을 그대로 재정의 → 위에서 정의한 gpt(), 학습/추론 루프가 자동으로 새 데이터를 사용
docs = [decompose(t) for t in MOVIE_TITLES]
uchars = sorted({j for d in docs for j in d})
BOS = len(uchars)
vocab_size = len(uchars) + 1
block_size = 24                                    # 영화 제목이 더 길어서 12 → 24

lens = [len(d) for d in docs]
print(f'영화 제목 수    : {len(docs)}편')
print(f'자모 길이      : 최소 {min(lens)}, 최대 {max(lens)}, 평균 {sum(lens)/len(lens):.1f}')
print(f'unique 자모    : {len(uchars)}개  → vocab_size {vocab_size}')
print(f'block_size     : {block_size} (이름 학습은 12, 영화는 24)')
print()
print('예시 분해:')
for title, doc in list(zip(MOVIE_TITLES, docs))[:5]:
    print(f'  {title:8s} ({len(doc):2d} jamo) → {doc}')

In [ ]:
# 9-2. state_dict 재초기화 (n_layer / n_embd / n_head 는 그대로 유지)
# matrix lambda는 §4에서 정의된 것을 그대로 사용
state_dict = {
    'wte':     matrix(vocab_size, n_embd),
    'wpe':     matrix(block_size, n_embd),
    'lm_head': matrix(vocab_size, n_embd),
}
for i in range(n_layer):
    state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)
params = [p for mat in state_dict.values() for row in mat for p in row]

print(f'num params (영화) : {len(params)}')
print(f'  ↑ vocab/block_size이 커져서 이름 학습 때보다 wte/wpe/lm_head가 살짝 큼')

In [ ]:
# 9-3. 학습 루프 — §7과 코드가 한 줄도 다르지 않음 (확인용으로 비교해보세요).
#       이름 학습 때 쓴 같은 코드가 docs/uchars/BOS/vocab_size/block_size/state_dict/params 가
#       위에서 새로 바인딩되었기 때문에 자동으로 영화 데이터를 학습합니다.
learning_rate, beta1, beta2, eps_adam = 0.01, 0.85, 0.99, 1e-8
m = [0.0] * len(params)
v = [0.0] * len(params)

num_steps = 300   # 시퀀스가 더 길어 step당 시간 약간 증가. 결과가 약하면 600 / 1000으로
loss_history_movie = []

for step in range(num_steps):
    doc = docs[step % len(docs)]
    tokens = [BOS] + [uchars.index(j) for j in doc] + [BOS]
    n = min(block_size, len(tokens) - 1)

    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    losses = []
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits)
        losses.append(-probs[target_id].log())
    loss = (1 / n) * sum(losses)
    loss.backward()

    lr_t = learning_rate * (1 - step / num_steps)
    for i, p in enumerate(params):
        m[i] = beta1 * m[i] + (1 - beta1) * p.grad
        v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
        m_hat = m[i] / (1 - beta1 ** (step + 1))
        v_hat = v[i] / (1 - beta2 ** (step + 1))
        p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
        p.grad = 0

    loss_history_movie.append(loss.data)
    if (step + 1) % 20 == 0 or step == 0:
        print(f'step {step+1:4d} / {num_steps:4d} | loss {loss.data:.4f}')

print('\n영화 학습 완료!')

In [ ]:
# 9-4. 추론: 영화 제목 생성 — §8의 이름 추론 코드와 거의 동일
#       (성씨 추첨 트릭이 없으므로 제목 전체를 BOS→...→BOS 로 모델이 직접 생성)
temperature = 0.7
n_samples   = 30
train_set   = set(MOVIE_TITLES)

print(f'--- 영화 제목 생성 (temperature={temperature}, {n_samples}개) ---\n')
results = []
for sample_idx in range(n_samples):
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    token_id = BOS
    sample = []
    for pos_id in range(block_size):
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax([l / temperature for l in logits])
        token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
        if token_id == BOS:
            break
        sample.append(uchars[token_id])
    title = compose(sample)
    results.append(title)

    is_valid = bool(title) and all((0xAC00 <= ord(c) < 0xD7A4) or c.isdigit() or c.isspace() for c in title)
    is_novel = is_valid and (title not in train_set)
    tag = '[NEW]' if is_novel else ('[OK ]' if is_valid else '[BAD]')
    print(f'  {sample_idx+1:2d}: {tag} {title}')

# 평가 지표
n_valid = sum(1 for t in results if t and all((0xAC00 <= ord(c) < 0xD7A4) or c.isdigit() or c.isspace() for c in t))
n_novel = sum(1 for t in results if t and t not in train_set and all((0xAC00 <= ord(c) < 0xD7A4) or c.isdigit() or c.isspace() for c in t))
print(f'\n=== 학습 정도 측정 ===')
print(f'유효 한글       : {n_valid}/{n_samples}  ({n_valid*100//n_samples}%)  ← 음절 구조 학습')
print(f'신조 제목 [NEW] : {n_novel}/{n_samples}  ({n_novel*100//n_samples}%)  ← 새 조합 생성 능력')

### 관찰 포인트

- **모델 코드**: `Value`, `linear`, `softmax`, `rmsnorm`, `gpt()`, 학습/추론 루프 — **한 줄도 안 바뀌었음**.
- **바뀐 것**: `docs`, `uchars`, `BOS`, `vocab_size`, `block_size`, `state_dict`, `params`. 즉 데이터와 그에 따른 모양뿐.
- **결과**: 같은 알고리즘이 영화 제목 도메인의 패턴(끝맺음 `-들/-자/-기/-도시`, 음운 조합 등)을 학습.

### 추가 실험
- `MOVIE_TITLES`를 본인 도메인으로 교체 — 음식명 / K-pop 곡명 / 지명 / 캐릭터 / 책 제목 등.
- `n_layer=2`, `num_steps=1000` 으로 키워서 [NEW] 비율 변화 측정.
- 학습 데이터를 ~160편으로 2배 늘렸을 때 vocab과 loss 곡선이 어떻게 변하는지 비교.

### Karpathy 원본과의 차이 요약
- **원본**: 영문 이름(`names.txt`, `\n`-구분 영문 문자열) + `uchars = sorted(set(''.join(docs)))` 직접 토큰화 + `block_size=16`.
- **본 노트북 §1–8**: 한국 이름 + 자모 분해 트릭(한글 음절 폭발 회피) + `block_size=12`.
- **본 섹션 §9**: 한국 영화 제목 + 같은 자모 트릭 + `block_size=24`.

> 어느 경우든 **모델 아키텍처(`gpt()` 함수)와 학습 루프는 동일**. 데이터/토큰화/컨텍스트 길이만 도메인에 맞게 재바인딩.


---
## 🔟 사전학습 vs scratch — "왜 LLaMA는 사전학습된 걸 받아쓰는가"

§9에서 영화 제목 80편만으로 학습하면 한글 음절 구조조차 잘 못 잡고 [BAD]가 많이 나옵니다.
이번 섹션은 그 한계를 **사전학습(pre-training) → 파인튜닝(fine-tuning)** 사이클로 극복합니다.

### 시나리오

| 실험 | pre-training | fine-tuning (영화) | 비고 |
|------|--------------|--------------------|------|
| **A. Scratch** | ❌ 없음 | ✅ 영화 80편으로 N step | 영화로만 학습 |
| **B. Pre-train + Fine-tune** | ✅ **한국어 일반 문장** ~80개로 M step | ✅ 영화 80편으로 **같은 N step** | 사전학습 후 파인튜닝 |

### 공정 비교를 위한 설계
- **동일 seed로 fresh state_dict 2번 생성** → 실험 A, B의 초기 weight가 완전히 동일
- **동일한 fine-tuning step 수** → 영화 학습 비용은 같음
- **동일 generation seed** → 같은 sampling 스트림에서 weight 차이만 결과에 반영
- 차이는 오직: **B는 영화 학습 전에 한국어 일반 텍스트로 한 번 더 학습받았다**

### 가설 (슬라이드 p.12 LLaMA 메시지의 미니 시연)
> 사전학습으로 음절 구조·jamo 전이 확률·자주 쓰이는 어휘 패턴을 미리 익힌 모델은,
> 같은 fine-tune step 수에서 **더 낮은 loss**, **더 높은 유효 한글 비율**을 달성한다.


In [ ]:
# 10-1. 한국어 일반 corpus (사전학습용) — microGPT 정신대로 외부 다운로드 없이 직접 박아둠
KOREAN_CORPUS = [
    # 인사
    '안녕하세요', '반갑습니다', '잘 부탁드립니다', '감사합니다',
    '고맙습니다', '죄송합니다', '미안해요', '괜찮아요',
    # 날씨/계절
    '오늘 날씨가 좋다', '비가 내린다', '바람이 분다', '눈이 쌓인다',
    '하늘이 맑다', '구름이 많다', '봄이 왔다', '여름이 뜨겁다',
    '가을 잎이 곱다', '겨울이 춥다',
    # 일상 행동
    '학교에 간다', '집으로 돌아온다', '잠을 잔다', '아침에 일어난다',
    '책을 읽는다', '음악을 듣는다', '영화를 본다',
    '운동을 한다', '산책을 나간다', '청소를 한다', '빨래를 한다',
    '요리를 한다', '편지를 쓴다', '전화를 받는다',
    # 식사
    '아침을 먹는다', '점심을 먹었다', '저녁을 먹자',
    '밥이 따뜻하다', '국이 맛있다', '김치가 시원하다', '과일이 달다',
    '커피를 마신다', '차를 우린다', '물을 마신다',
    # 감정
    '기분이 좋다', '마음이 편하다', '즐거운 하루다', '행복하다',
    '피곤하다', '슬프다', '심심하다', '걱정이 된다',
    # 장소/사람
    '공원에 갔다', '도서관에 들렀다', '시장을 둘러봤다', '바다를 봤다',
    '산을 올랐다', '카페에 앉았다', '식당에서 먹었다',
    '엄마가 웃는다', '아빠가 일하신다', '동생이 잔다', '친구를 만났다',
    '선생님이 가르치신다', '할머니께 인사드렸다',
    # 자연/동물
    '꽃이 피었다', '나무가 크다', '풀이 푸르다', '돌이 단단하다',
    '강이 흐른다', '바다가 넓다', '하늘이 높다',
    '별이 반짝인다', '달이 둥글다', '해가 뜬다',
    '고양이가 운다', '강아지가 뛴다', '새가 날아간다', '나비가 춤춘다',
    # 시간/추상
    '시간이 빠르다', '내일이 기대된다', '어제는 즐거웠다',
    '꿈을 꾸었다', '약속을 지켰다', '비밀을 알았다',
    '희망이 있다', '용기를 냈다', '도움을 받았다',
    # 학습/일
    '문제를 풀었다', '답을 찾았다', '시험을 봤다', '점수가 올랐다',
    '발표를 했다', '회의를 했다', '계획을 세웠다',
]

# §9의 MOVIE_TITLES 를 그대로 사용 (사전 실행 필수)
docs_corpus = [decompose(s) for s in KOREAN_CORPUS]
docs_movies = [decompose(t) for t in MOVIE_TITLES]

# 통합 vocab — corpus ∪ movies (양쪽 모두에 등장하는 자모를 한 번에 잡음)
uchars = sorted({j for d in (docs_corpus + docs_movies) for j in d})
BOS = len(uchars)
vocab_size = len(uchars) + 1
block_size = 24

print(f'pre-train corpus : {len(docs_corpus)}문장 (자모 평균 {sum(len(d) for d in docs_corpus)/len(docs_corpus):.1f})')
print(f'fine-tune movies : {len(docs_movies)}편   (자모 평균 {sum(len(d) for d in docs_movies)/len(docs_movies):.1f})')
print(f'통합 vocab_size  : {vocab_size}  (corpus∪movies)')
print(f'block_size       : {block_size}')

In [ ]:
# 10-2. 헬퍼 함수 — fresh weight 생성 / 학습 루프 / 평가용 생성
beta1, beta2, eps_adam = 0.85, 0.99, 1e-8

def make_fresh_state(seed=42):
    """동일 seed로 새 state_dict + params를 만들어 실험 A, B의 초기 weight를 동일하게."""
    random.seed(seed)
    sd = {
        'wte':     matrix(vocab_size, n_embd),
        'wpe':     matrix(block_size, n_embd),
        'lm_head': matrix(vocab_size, n_embd),
    }
    for i in range(n_layer):
        sd[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)
        sd[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)
        sd[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)
        sd[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)
        sd[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd)
        sd[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)
    ps = [p for mat in sd.values() for row in mat for p in row]
    return sd, ps

def train_loop(docs_, num_steps_, lr=0.01, label=''):
    """§7과 동일한 Adam 학습 루프. global state_dict, params 사용."""
    losses = []
    m = [0.0] * len(params)
    v = [0.0] * len(params)
    for step in range(num_steps_):
        doc = docs_[step % len(docs_)]
        tokens = [BOS] + [uchars.index(j) for j in doc] + [BOS]
        n = min(block_size, len(tokens) - 1)
        keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
        ls = []
        for pos_id in range(n):
            token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
            logits = gpt(token_id, pos_id, keys, values)
            probs = softmax(logits)
            ls.append(-probs[target_id].log())
        loss = (1 / n) * sum(ls)
        loss.backward()
        lr_t = lr * (1 - step / num_steps_)
        for i, p in enumerate(params):
            m[i] = beta1 * m[i] + (1 - beta1) * p.grad
            v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
            m_hat = m[i] / (1 - beta1 ** (step + 1))
            v_hat = v[i] / (1 - beta2 ** (step + 1))
            p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
            p.grad = 0
        losses.append(loss.data)
        if (step + 1) % 50 == 0 or step == 0:
            print(f'  [{label}] step {step+1:4d}/{num_steps_} | loss {loss.data:.4f}')
    return losses

def gen_and_eval(temperature=0.7, n_samples=30, gen_seed=123):
    """같은 gen_seed로 sampling stream을 통일 → 모델 weight 차이만 결과에 반영."""
    random.seed(gen_seed)
    train_set = set(MOVIE_TITLES)
    results = []
    for _ in range(n_samples):
        keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
        token_id = BOS
        sample = []
        for pos_id in range(block_size):
            logits = gpt(token_id, pos_id, keys, values)
            probs = softmax([l / temperature for l in logits])
            token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
            if token_id == BOS:
                break
            sample.append(uchars[token_id])
        results.append(compose(sample))
    n_valid = sum(1 for t in results if t and all((0xAC00 <= ord(c) < 0xD7A4) or c.isdigit() or c.isspace() for c in t))
    n_novel = sum(1 for t in results if t and t not in train_set and all((0xAC00 <= ord(c) < 0xD7A4) or c.isdigit() or c.isspace() for c in t))
    return results, n_valid, n_novel

# 실험 step 수 — 시간 절약을 위해 default는 보수적. 신호를 더 또렷이 보려면 2배로 늘리세요.
N_FINETUNE = 100   # 영화 fine-tune step (실험 A, B 동일)
M_PRETRAIN = 200   # 한국어 일반 corpus pre-train step (실험 B 만)
print(f'\n실험 설정: pre-train {M_PRETRAIN} step, fine-tune {N_FINETUNE} step')
print('(순수 Python autograd라 약 15~25분 소요. 더 또렷한 신호를 원하면 위 값을 2배로)')

### 10-A. 실험 A — Scratch (영화로만 학습)

`make_fresh_state(seed=42)` 로 초기화된 모델을 영화 제목 80편으로 `N_FINETUNE` step만 학습합니다.


In [ ]:
# 실험 A: 영화로만 N_FINETUNE step
print('=== 실험 A: Scratch — 영화로만 학습 ===')
state_dict, params = make_fresh_state(seed=42)
loss_A = train_loop(docs_movies, N_FINETUNE, label='A:scratch')
samples_A, valid_A, novel_A = gen_and_eval()

print(f'\n[실험 A 결과] 최종 loss {loss_A[-1]:.3f} | 유효 한글 {valid_A}/30 | 신조 [NEW] {novel_A}/30')
print('샘플 (앞 12개):')
for i, s in enumerate(samples_A[:12], 1):
    is_valid = bool(s) and all((0xAC00 <= ord(c) < 0xD7A4) or c.isdigit() or c.isspace() for c in s)
    print(f'  {i:2d}: {"[OK ]" if is_valid else "[BAD]"} {s}')

### 10-B. 실험 B — Pre-train (한국어 일반) → Fine-tune (영화)

**같은 seed=42** 로 다시 fresh init 한 다음, 한국어 일반 문장 ~80개로 `M_PRETRAIN` step 사전학습 → 그 weight 위에서 영화 80편으로 **`N_FINETUNE` step 파인튜닝** (실험 A와 같은 step 수).

> 핵심: A와 B의 초기 weight는 동일. **차이는 영화 학습 전에 한국어를 한 번 본 적 있느냐 뿐.**


In [ ]:
# 실험 B: 동일 seed로 fresh init → pre-train (한국어 일반) → fine-tune (영화 N_FINETUNE)
print('=== 실험 B: Pre-train (한국어 일반) → Fine-tune (영화) ===')
state_dict, params = make_fresh_state(seed=42)   # ← A와 완전히 동일한 초기 weight

print('\n[B-1] 사전학습 단계 — 한국어 일반 문장으로 일반 패턴 습득')
loss_pretrain = train_loop(docs_corpus, M_PRETRAIN, label='B:pretrain')

print('\n[B-2] 파인튜닝 단계 — 사전학습된 weight 위에서 영화 도메인 specialize')
loss_finetune = train_loop(docs_movies, N_FINETUNE, label='B:finetune')

samples_B, valid_B, novel_B = gen_and_eval()

print(f'\n[실험 B 결과] 최종 loss {loss_finetune[-1]:.3f} | 유효 한글 {valid_B}/30 | 신조 [NEW] {novel_B}/30')
print('샘플 (앞 12개):')
for i, s in enumerate(samples_B[:12], 1):
    is_valid = bool(s) and all((0xAC00 <= ord(c) < 0xD7A4) or c.isdigit() or c.isspace() for c in s)
    print(f'  {i:2d}: {"[OK ]" if is_valid else "[BAD]"} {s}')

In [ ]:
# 10-C. 비교: loss 곡선 + 정량 지표
import matplotlib.pyplot as plt
import math

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 사전학습 loss (B만) — corpus에서 일반 패턴을 학습하는 모습
ax1.plot(loss_pretrain, color='steelblue', linewidth=1.5)
ax1.axhline(y=math.log(vocab_size), color='gray', linestyle='--', alpha=0.6,
            label=f'uniform baseline = {math.log(vocab_size):.2f}')
ax1.set_xlabel('Pre-training Step (Korean general corpus)')
ax1.set_ylabel('Loss')
ax1.set_title(f'[B] Pre-training Phase ({M_PRETRAIN} steps)')
ax1.legend()
ax1.grid(alpha=0.3)

# 오른쪽: 영화 fine-tune A vs B (같은 step 수, 같은 데이터, 같은 초기 random seed)
ax2.plot(loss_A,        color='crimson', linewidth=1.5, label=f'A: scratch (final {loss_A[-1]:.3f})')
ax2.plot(loss_finetune, color='green',   linewidth=1.5, label=f'B: pre-trained → fine-tune (final {loss_finetune[-1]:.3f})')
ax2.set_xlabel('Movie Fine-tuning Step')
ax2.set_ylabel('Loss')
ax2.set_title(f'Movie Training Loss: Scratch vs Pre-trained ({N_FINETUNE} steps each)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 정량 비교표
print('\n' + '='*70)
print(f'{"":>30s}  {"유효 한글":>10s}  {"신조 [NEW]":>10s}  {"최종 loss":>10s}')
print('-' * 70)
print(f'{"A. Scratch (영화로만)":>30s}  {valid_A:>5d}/30   {novel_A:>5d}/30   {loss_A[-1]:>10.3f}')
print(f'{"B. Pre-train + Fine-tune":>30s}  {valid_B:>5d}/30   {novel_B:>5d}/30   {loss_finetune[-1]:>10.3f}')
print('=' * 70)
delta_valid = valid_B - valid_A
delta_loss  = loss_A[-1] - loss_finetune[-1]
print(f'개선폭 (B - A)              : 유효 한글 +{delta_valid:>3d}개   |   loss -{delta_loss:.3f}')
if delta_valid > 0 or delta_loss > 0.1:
    print('\n→ 사전학습이 같은 fine-tune 비용으로 더 좋은 출발점을 제공함을 확인.')
else:
    print('\n→ 신호가 약함. M_PRETRAIN을 400~600으로, N_FINETUNE을 200으로 늘려보세요.')

### 10-D. 결론 — "왜 LLaMA를 그냥 받아쓰는가"

이번 실험은 슬라이드 p.12의 **microGPT(~4M params) vs LLaMA-3.2-3B (3B params, 750배)** 비교의 **사전학습 측면**을 미니로 재현한 것입니다.

| 항목 | microGPT 실험 (이 노트북) | 실제 LLaMA |
|------|---------------------------|-----------|
| 사전학습 corpus | 한국어 일반 문장 ~80개 | 인터넷 텍스트 **수조 토큰** |
| 사전학습 step | M_PRETRAIN (200~600) | 수십만 ~ 수백만 step (수개월, 수만 GPU) |
| 파인튜닝 비용 | N_FINETUNE × 영화 80편 | 도메인 데이터셋 + 며칠 |
| 결론 | 동일 fine-tune step에서 B가 A보다 좋음 | **사전학습된 weight를 받아쓰는 게 압도적으로 효율적** |

### 핵심 메시지
- 사전학습은 **음절 구조 / jamo 전이 / 자주 쓰이는 어휘 패턴** 같은 "한국어로서 자연스러운 다음 토큰 분포"의 prior 를 학습해둠.
- 이 prior 위에서 도메인(영화) 학습을 하면 **같은 fine-tune step**으로도 더 낮은 loss·더 높은 유효 한글 비율 달성.
- 이게 바로 실제 산업이 **GPT-4 / LLaMA 같은 사전학습된 모델을 받아 fine-tuning만 하는 이유** — 사전학습은 한 번 하면 수많은 다운스트림 태스크가 그 prior를 공짜로 누림.

### 추가 실험
- `M_PRETRAIN` 을 400, 600, 800 으로 늘려가며 차이가 얼마나 벌어지는지 측정 (scaling law의 미니 버전).
- `KOREAN_CORPUS` 를 **영화와 무관한 도메인** (수학 문제, 동요 가사, 고전소설 일부) 으로 바꿔도 사전학습 효과가 있는지 — "사전학습 데이터의 도메인 적합성"이 얼마나 중요한지 체감.
- `N_FINETUNE = 0` 으로 두고 사전학습만 한 모델로 영화 제목을 생성해보기 — "general 한국어 모델"이 아무 학습 없이도 영화 비슷한 걸 만들 수 있는지.


---
## 정리

### 우리가 이번 노트북에서 만든 것

| 구성요소 | 역할 | 외부 라이브러리 |
|----------|------|-----------------|
| `decompose` / `compose` | 한글 ↔ 자모 변환 (Unicode 산술만 사용) | 직접 구현 |
| `Value` 클래스 | scalar autograd (chain rule) | 직접 구현 |
| `linear`, `softmax`, `rmsnorm` | Transformer 빌딩 블록 | 직접 구현 |
| `gpt(...)` | embedding + MHA + MLP + residual | 직접 구현 |
| Adam optimizer | m / v 모멘트, bias correction, lr decay | 직접 구현 |
| 학습/추론 루프 | next-jamo 예측 → 자모 시퀀스 샘플링 → 한글 합성 | 직접 구현 |

### 무엇을 학습시켰는가

- **태스크**: GPT가 한국 이름의 자모 패턴을 학습하는 next-token LM
- **데이터**: ~3000개 한국 이름 (성씨 37 × 이름 80 조합), 자모로 분해
- **vocab**: 데이터에 등장한 자모만 (~30), microGPT 스케일에 맞춤
- **결과 평가 방법**:
  1. 출력 자모가 *유효한 한글 음절* 로 합성되는지 (구조 학습)
  2. 합성된 한글이 *이름같은가* (분포 학습)
  3. 학습 데이터에 *없던 새 이름* 이 생성되는가 (일반화)

### minGPT vs microGPT vs GPT-2

| 항목 | Karpathy minGPT | Karpathy microGPT | OpenAI GPT-2 |
|------|-----------------|-------------------|--------------|
| 발표 | 2020 | 2026.02 | 2019 |
| 코드 줄 수 | ~300 | **243** | 수천 |
| 의존성 | PyTorch | **없음 (순수 Python)** | TensorFlow |
| 목적 | 교육용 PyTorch 구현 | **알고리즘 본질 시각화** | production |
| 본 노트북 적용 | — | 한국 이름 자모 LM (~1만 파라미터) | — |

### 핵심 학습 포인트

1. **GPT는 마법이 아니다** — 243줄 파이썬으로 알고리즘 전체가 표현된다.
2. **Autograd는 토폴로지 정렬 + chain rule 한 줄** — 30줄이면 충분하다.
3. **언어 의존적이지 않다** — 영문 이름이든 한글 자모든, 토큰만 정의되면 동일 알고리즘이 패턴을 학습한다.
4. **자모 분해 트릭** 으로 한글의 vocab 폭발 문제를 우회 → 초경량 모델로도 한국어 처리 가능.
5. PyTorch가 하는 일은 **이걸 GPU에서 빠르게** 돌리는 것뿐이다 ("Everything else is just efficiency").

### 추가 실습 아이디어

- `n_layer`, `n_embd` 를 키워서 더 다양한 이름 생성
- `SURNAMES` / `GIVEN_NAMES` 리스트를 본인 가족/지인 이름으로 교체
- 한국 지명 (서울, 부산, 대전, ...) 또는 음식명 (김치, 비빔밥, ...) 으로 도메인 변경
- 두 글자 이름 → 세 글자 이름으로 확장 (block_size 키워야 함)
- 자모 단위 → **음절 단위** (vocab 100개 정도) 로 바꿨을 때 학습 차이 비교
- 영문 + 한글 혼합 (e.g. K-pop 가수명 `BTS 정국`) 의 vocab 처리

In [ ]:
print('Session 7 완료!')
print('다음 세션(Session 8): 합성 데이터 생성 & Knowledge Distillation')